# Merfish Mouse Data

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import os
import re
import random
from sklearn.metrics import adjusted_rand_score

import scSLAT
from scSLAT.model import Cal_Spatial_Net, load_anndatas, run_SLAT, spatial_match
from scSLAT.viz import match_3D_multi, hist, Sankey

import warnings
warnings.filterwarnings("ignore")

## Load Data

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # If using multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False  # May slow down training


def cal_laa_ari(ad_df):
    true_clusters_mapped_region = []  # Define lists inside the function
    mapped_clusters_mapped_region = []
    cnt = 0  
    
    for i in range(len(ad_df)):
        f1_cluster = ad_df.iloc[i]['target_celltype']
        f2_cluster = ad_df.iloc[i]['celltype']
        true_clusters_mapped_region.append(f1_cluster)
        mapped_clusters_mapped_region.append(f2_cluster)
        if f1_cluster == f2_cluster:
            cnt += 1 

    laa = cnt/len(ad_df)
    ari = adjusted_rand_score(true_clusters_mapped_region, mapped_clusters_mapped_region)
              
    return laa,ari

In [3]:
files = ['merfish_mouse_brain_slice1.h5ad','merfish_mouse_brain_slice2.h5ad','merfish_mouse_brain_slice3.h5ad','merfish_mouse_brain_slice4.h5ad',
'merfish_mouse_brain_slice5.h5ad','merfish_mouse_brain_slice6.h5ad','merfish_mouse_brain_slice7.h5ad','merfish_mouse_brain_slice8.h5ad',
'merfish_mouse_brain_slice9.h5ad','merfish_mouse_brain_slice10.h5ad','merfish_mouse_brain_slice11.h5ad','merfish_mouse_brain_slice12.h5ad',
'merfish_mouse_brain_slice13.h5ad','merfish_mouse_brain_slice14.h5ad','merfish_mouse_brain_slice15.h5ad','merfish_mouse_brain_slice16.h5ad',
'merfish_mouse_brain_slice17.h5ad','merfish_mouse_brain_slice18.h5ad','merfish_mouse_brain_slice19.h5ad','merfish_mouse_brain_slice20.h5ad',
'merfish_mouse_brain_slice21.h5ad','merfish_mouse_brain_slice22.h5ad','merfish_mouse_brain_slice23.h5ad','merfish_mouse_brain_slice24.h5ad',
'merfish_mouse_brain_slice25.h5ad','merfish_mouse_brain_slice26.h5ad','merfish_mouse_brain_slice27.h5ad','merfish_mouse_brain_slice28.h5ad',
'merfish_mouse_brain_slice29.h5ad','merfish_mouse_brain_slice30.h5ad','merfish_mouse_brain_slice31.h5ad','merfish_mouse_brain_slice32.h5ad']

In [ ]:
seeds_to_run = [2024, 1, 2, 3, 4, 10, 11, 12, 13, 14]

# seeds_to_run = [2024, 1]
for i in range(len(files)-1):
    file_path = f'/pkl_data/{files[i]}'
    file_path2 = f'/pkl_data/{files[i+1]}'
    ad = sc.read(file_path)
    ad_target = sc.read(file_path2)
    
    ad_target.obsm['spatial_3D'] = ad_target.obsm['spatial']
    ad_target.obsm['spatial'] = ad_target.obsm['spatial'].to_numpy()[:, :2]  # Use 2D for plotting
    ad.obsm['spatial_3D'] = ad.obsm['spatial']
    ad.obsm['spatial'] = ad.obsm['spatial'].to_numpy()[:, :2]  # Use 2D for plotting
    ad.obs['original_clusters']=ad.obs['spa_cluster'].astype('category')
    ad_target.obs['original_clusters']=ad_target.obs['spa_cluster'].astype('category')
    
    for seed in seeds_to_run:
        print('--------------'+ str(seed))
        set_seed(seed)

        # Calculating spatial neighbor graph
        Cal_Spatial_Net(ad_target, k_cutoff=10, model='KNN')
        Cal_Spatial_Net(ad, k_cutoff=10, model='KNN')

        # Use DPCA feature to format graph
        edges, features = load_anndatas([ad_target, ad], feature='DPCA', check_order=False)
        embd0, embd1, time = run_SLAT(features, edges)
        best, index, distance = spatial_match([embd0, embd1], adatas=[ad_target, ad], reorder=False)

        ad_target_df = pd.DataFrame({'index': range(embd0.shape[0]),
                                'x': ad_target.obsm['spatial'][:,0],
                                'y': ad_target.obsm['spatial'][:,1],
                                'celltype': ad_target.obs['original_clusters']})
        ad_df = pd.DataFrame({'index': range(embd1.shape[0]),
                                'x': ad.obsm['spatial'][:,0],
                                'y': ad.obsm['spatial'][:,1],
                                'celltype': ad.obs['original_clusters']})

        matching = np.array([range(index.shape[0]), best])
        best_match = distance[:,0]

        ad_df['target_index'] = best
        ad_df['target_celltype'] = ad_target_df.iloc[matching[1,:],:]['celltype'].to_list()
        ad_df['cos_similarity'] = best_match
        
        number = re.search(r'\d+', files[i]).group()
        number2 = re.search(r'\d+', files[i+1]).group()
        ad_df.to_csv(f'/pkl_data/{seed}/merfish_{number}.csv', index=False)

        laa,ari = cal_laa_ari(ad_df)
        print(f'LAA: {laa}')
        print(f'ARI: {ari}')

        column = f'{number}_{number2}'
        filename = 'merfish_ARI.csv'
        excel_path = f'/pkl_data/{seed}/{filename}'
        header = [' ', '010_011']

        # Check if the file exists
        if not os.path.exists(excel_path):
            # Define the header and first row
            first_row = [[0, ari]]
            # Create a DataFrame
            df = pd.DataFrame(first_row, columns=header)
            print(df)
            # Save as a CSV file
            df.to_csv(excel_path, index=False)
            
        filename = 'merfish_LAA.csv'
        excel_path = f'/pkl_data/{seed}/{filename}'
        # Check if the file exists
        if not os.path.exists(excel_path):
            # Define the header and first row
            first_row = [[0, laa]]
            # Create a DataFrame
            df = pd.DataFrame(first_row, columns=header)
            print(df)
            # Save as a CSV file
            df.to_csv(excel_path, index=False)
            
        filename = 'mHemi_brain_RunTime.xlsx'
        excel_path = f'/pkl_data/{seed}/{filename}'
        # Check if the file exists
        if not os.path.exists(excel_path):
            # Define the header and first row
            first_row = [[0, time]]
            # Create a DataFrame
            df = pd.DataFrame(first_row, columns=header)
            print(df)
            # Save as a CSV file
            df.to_excel(excel_path, index=False)

In [6]:
seeds_to_run = [2024, 1, 2, 3, 4, 10, 11, 12, 13, 14]

# seeds_to_run = [2024, 1]
for i in range(len(files)-1):
    file_path = f'/pkl_data/{files[i]}'
    file_path2 = f'/pkl_data/{files[i+1]}'
    ad = sc.read(file_path)
    ad_target = sc.read(file_path2)
    
    ad_target.obsm['spatial_3D'] = ad_target.obsm['spatial']
    ad_target.obsm['spatial'] = ad_target.obsm['spatial'].to_numpy()[:, :2]  # Use 2D for plotting
    ad.obsm['spatial_3D'] = ad.obsm['spatial']
    ad.obsm['spatial'] = ad.obsm['spatial'].to_numpy()[:, :2]  # Use 2D for plotting
    ad.obs['original_clusters']=ad.obs['spa_cluster'].astype('category')
    ad_target.obs['original_clusters']=ad_target.obs['spa_cluster'].astype('category')
    
    for seed in seeds_to_run:
        print('--------------'+ str(seed))
        set_seed(seed)

        # Calculating spatial neighbor graph
        Cal_Spatial_Net(ad_target, k_cutoff=10, model='KNN')
        Cal_Spatial_Net(ad, k_cutoff=10, model='KNN')

        # Use DPCA feature to format graph
        edges, features = load_anndatas([ad_target, ad], feature='DPCA', check_order=False)
        embd0, embd1, time = run_SLAT(features, edges)
        best, index, distance = spatial_match([embd0, embd1], adatas=[ad_target, ad], reorder=False)

        ad_target_df = pd.DataFrame({'index': range(embd0.shape[0]),
                                'x': ad_target.obsm['spatial'][:,0],
                                'y': ad_target.obsm['spatial'][:,1],
                                'celltype': ad_target.obs['original_clusters']})
        ad_df = pd.DataFrame({'index': range(embd1.shape[0]),
                                'x': ad.obsm['spatial'][:,0],
                                'y': ad.obsm['spatial'][:,1],
                                'celltype': ad.obs['original_clusters']})

        matching = np.array([range(index.shape[0]), best])
        best_match = distance[:,0]

        ad_df['target_index'] = best
        ad_df['target_celltype'] = ad_target_df.iloc[matching[1,:],:]['celltype'].to_list()
        ad_df['cos_similarity'] = best_match
        
        number = re.search(r'\d+', files[i]).group()
        number2 = re.search(r'\d+', files[i+1]).group()
        ad_df.to_csv(f'/pkl_data/{seed}/merfish_{number}.csv', index=False)

        laa,ari = cal_laa_ari(ad_df)
        print(f'LAA: {laa}')
        print(f'ARI: {ari}')

        column = f'{number}_{number2}'
        filename = 'merfish_ARI.csv'
        excel_path = f'=/pkl_data/{seed}/{filename}'
        # Load existing file
        df = pd.read_csv(excel_path)
        df.at[0, column] = ari
        print(df)
        # Save as a CSV file
        df.to_csv(excel_path, index=False)

        filename = 'merfish_LAA.csv'
        excel_path = f'/pkl_data/{seed}/{filename}'
        # Load existing file
        df = pd.read_csv(excel_path)
        df.at[0, column] = laa
        print(df)
        # Save as a CSV file
        df.to_csv(excel_path, index=False)

        filename = 'merfish_RunTime.xlsx'
        excel_path = f'/pkl_data/{seed}/{filename}'
        # Load existing file
        df = pd.read_excel(excel_path, engine='openpyxl')
        df.at[0, column] = time
        print(df)
        # Save as a CSV file
        df.to_excel(excel_path, index=False)

--------------2024
Calculating spatial neighbor graph ...
The graph contains 23035 edges, 2033 cells.
11.33054599114609 neighbors per cell on average.
Calculating spatial neighbor graph ...
The graph contains 27333 edges, 2423 cells.
11.280643829962855 neighbors per cell on average.
Use DPCA feature to format graph
Warning! Dual PCA is using GPU, which may lead to OUT OF GPU MEMORY in big dataset!
Choose GPU:0 as device
Running
Training model time: 0.53
LAA: 0.5617003714403632
ARI: 0.3174901322789482
           0_1      1_2
0  0  0.599752  0.31749
           0_1     1_2
0  0  0.678722  0.5617
           0_1      1_2
0  0  0.545769  0.52916
--------------1
Calculating spatial neighbor graph ...
The graph contains 23035 edges, 2033 cells.
11.33054599114609 neighbors per cell on average.
Calculating spatial neighbor graph ...
The graph contains 27333 edges, 2423 cells.
11.280643829962855 neighbors per cell on average.
Use DPCA feature to format graph
Warning! Dual PCA is using GPU, which 